# 11. Statistics and Confidence Intervals

This notebook adds inferential statistics to the study. Single summary metrics are insufficient for strong claims, particularly when model comparisons are close. The preferred approach is to evaluate variability across seeds and to bootstrap uncertainty intervals for per-sample, per-condition, or per-cell metrics.


In [1]:
!pip -q install anndata scanpy scikit-learn scipy pandas numpy torch pyarrow

from google.colab import drive
drive.mount('/content/drive')

import os
import sys
import urllib.request
from pathlib import Path

HELPER_DIR = Path("/content/drive/MyDrive/ChemDFM")
if str(HELPER_DIR) not in sys.path:
    sys.path.append(str(HELPER_DIR))

from chemdfm_notebook_helpers import *

DATA_PATH = Path("/content/drive/MyDrive/ChemDFM/data/raw/sciplex_complete_middle_subset.h5ad")
DATA_PATH.parent.mkdir(parents=True, exist_ok=True)

if not DATA_PATH.exists():
    print("Downloading SciPlex dataset...")
    URL = "https://f003.backblazeb2.com/file/chemCPA-datasets/sciplex_complete_middle_subset.h5ad"
    urllib.request.urlretrieve(URL, DATA_PATH)
    print("Download complete.")

print("PROJECT_ROOT =", PROJECT_ROOT)
print("DATA_PATH =", DATA_PATH)
print("Using device:", DEVICE)

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 176.6/176.6 kB 10.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.1/2.1 MB 71.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.1/60.1 kB 4.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 295.7/295.7 kB 21.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.2/9.2 MB 88.4 MB/s eta 0:00:00
Mounted at /content/drive
PROJECT_ROOT = /content/drive/MyDrive/ChemDFM
DATA_PATH = /content/drive/MyDrive/ChemDFM/data/raw/sciplex_complete_middle_subset.h5ad
Using device: cpu


In [2]:
import sys
from pathlib import Path

_support_candidates = [
    Path.cwd() / "notebooks_support",
    Path.cwd().parent / "notebooks_support",
    Path("/content/drive/MyDrive/ChemDFM/notebooks_support"),
]
for _cand in _support_candidates:
    if _cand.exists():
        sys.path.append(str(_cand))
        break

from chemdfm_notebook_helpers import *
print("Using device:", DEVICE)


Using device: cpu


In [3]:
STATS_DIR = RESULTS_DIR / "robustness"
STATS_DIR.mkdir(parents=True, exist_ok=True)

deg_summary_path = RESULTS_DIR / "canonical" / "deg_auprc_per_sample.csv"
dose_path = RESULTS_DIR / "canonical" / "dose_trend_groupwise.csv"

if deg_summary_path.exists():
    deg_df = pd.read_csv(deg_summary_path)
    ci_rows = []
    for keys, sub in deg_df.groupby(["split", "cell_type"]):
        ci = bootstrap_ci(sub["deg_auprc"].values, n_boot=1000, seed=SEED)
        ci_rows.append({"split": keys[0], "cell_type": keys[1], "metric": "deg_auprc", **ci})
    ci_df = pd.DataFrame(ci_rows)
    ci_df.to_csv(STATS_DIR / "metric_ci_summary.csv", index=False)
    display(ci_df.head())
else:
    print("Run 07_posthoc_gene_space_evaluation.ipynb first.")


,split,cell_type,metric,mean,lower,upper
0,ood,A549,deg_auprc,0.155668,0.152817,0.158195
1,ood,K562,deg_auprc,0.121474,0.119446,0.123557
2,ood,MCF7,deg_auprc,0.209336,0.206635,0.212108
3,test,A549,deg_auprc,0.150213,0.149182,0.151316
4,test,K562,deg_auprc,0.117918,0.117074,0.118730


In [4]:
# Seed-sweep aggregation scaffold
seed_records = []
for run_dir in RUNS_DIR.glob("*"):
    cfg_path = run_dir / "config_resolved.json"
    metrics_path = run_dir / "final_overall_metrics.csv"
    if cfg_path.exists() and metrics_path.exists():
        cfg = json.load(open(cfg_path))
        df = pd.read_csv(metrics_path)
        seed = cfg.get("seed", 42)
        model_name = df["model"].iloc[0] if "model" in df.columns else run_dir.name
        for _, row in df.iterrows():
            seed_records.append({
                "run_name": run_dir.name,
                "seed": seed,
                "split": row["split"],
                "model": model_name,
                "r2_top50": row.get("r2_top50", np.nan),
                "r2_full": row.get("r2_full", np.nan),
            })
seed_df = pd.DataFrame(seed_records)
seed_df.to_csv(STATS_DIR / "seed_sweep_summary.csv", index=False)
seed_df.head()


,run_name,seed,split,model,r2_top50,r2_full
0,residual_base,42,test,ResidualDoseResponseModel,0.577561,0.636585
1,residual_base,42,ood,ResidualDoseResponseModel,0.557015,0.614290
2,residual_cellaware_mmd,42,test,ResidualDoseResponseModel_CellAwareMMD,0.575434,0.635105
3,residual_cellaware_mmd,42,ood,ResidualDoseResponseModel_CellAwareMMD,0.558485,0.615382


For the final paper, each key claim should be paired with a variability estimate. At minimum, confidence intervals should be reported for overall `r2_top50`, DEG AUPRC, pathway-profile correlation, and dose-trend agreement. When multiple seeds are available, paired model differences should be summarized rather than relying only on raw means.
